# Lesson 06 · What does + mean?

numpy for sound, then our own types: classes, operators, pitches and scales.

### How to use this notebook

Run the cells in order. Each section starts from what the previous one built. The **Your turn** cells are for you: try before reading the answer below them.

**Recall:** arrays, `np.arange`, `np.zeros` and slices come from the numpy lab. Semitones and `2 ** (s / 12)` come from lessons 01–02, `%` from lesson 02, `render_events` and the pluck from lesson 05.

## 1. Setup

In [ ]:
from IPython.display import Audio
import numpy as np
from matplotlib import pyplot as plt

SR = 48000

## 2. numpy for sound

numpy's philosophy: an arithmetic operation on an array applies to **every element**.

In [ ]:
a = np.arange(10)
print(a)
print(a / 10)

One second of sound has `SR` samples. Dividing `np.arange(SR)` by `SR` gives the **time** of every sample, in seconds: 0, 1/48000, 2/48000, …

In [ ]:
t = np.arange(SR) / SR
print(t[:10])

`np.sin` is numpy's sine: given an array, it computes the sine of every element. With `t` as argument we get a sine evaluated at the time of every sample. Its period is 2π seconds, longer than the second we are plotting:

In [ ]:
plt.plot(np.sin(t))

Multiplying `t` by 2π gives one period per second: 1 Hz, too low to hear. Multiplying also by 440 gives 440 periods per second, A4. The plot becomes a block: 440 periods do not fit in a picture, but we can listen to them.

In [ ]:
plt.plot(np.sin(2 * np.pi * t))

In [ ]:
wave = np.sin(2 * np.pi * 440 * t)
Audio(wave, rate=SR)

To turn the tone into a note we need an **envelope**. A decaying exponential: `np.exp(-t / tau)` falls faster when `tau` (in seconds) is smaller.

In [ ]:
plt.plot(np.exp(-t / 0.3))

Lesson 05 needed a `vca` function to apply an envelope to a wave. With arrays it is a multiplication, sample by sample:

In [ ]:
env = np.exp(-t / 0.3)
sound = env * wave
plt.plot(sound)

The usual `play`, now one line, and a function for the whole note. `pluck` takes a frequency in Hz; `dur_s` is a duration in seconds. In this course a name ending in `_s` is in seconds.

In [ ]:
def play(wave):
    return Audio(wave, rate=SR, normalize=False, autoplay=False)

def pluck(f, dur_s=1.0, tau=0.3, amplitude=0.25):
    t = np.arange(round(dur_s * SR)) / SR
    env = np.exp(-t / tau)
    wave = np.sin(2 * np.pi * f * t)
    return amplitude * env * wave

play(pluck(440))

`amplitude * env * wave`: a number times an array multiplies every sample, so `amplitude` controls the volume.

### Sounds in time

A **sound** is a pair `(onset_s, wave)`: a waveform and the time, in seconds, at which it starts. In lesson 05 these pairs were called *events*; here they are *sounds*, because the word *event* will get a musical meaning in the next lesson.

`mix_sounds` puts every sound in its place:

1. the total length is where the last sound ends: `onset_s + len(wave) / SR`, the largest over all sounds;
2. `out` starts as silence, `np.zeros`;
3. each wave is added into its slice of `out`, starting at its onset converted to samples. `+=` on a slice adds sample by sample, as in the lab.

In [ ]:
def mix_sounds(sounds):
    dur_s = max([onset_s + len(wave) / SR for onset_s, wave in sounds])
    out = np.zeros(round(dur_s * SR))
    for onset_s, wave in sounds:
        onset = round(onset_s * SR)
        out[onset:onset + len(wave)] += wave
    return out

s1 = (0.0, pluck(440))
s2 = (0.25, pluck(330))
play(mix_sounds([s1, s2]))

`onset_s` is in seconds, `onset` in samples: the suffix keeps them apart.

### Your turn · A major scale

Play a major scale from A4 (440 Hz), one note every 0.25 s. The steps of a major scale, in semitones from the tonic: `[0, 2, 4, 5, 7, 9, 11, 12]`.

In [ ]:
scale = [0, 2, 4, 5, 7, 9, 11, 12]
sounds = [(i * 0.25, pluck(440 * 2 ** (scale[i] / 12))) for i in range(len(scale))]
play(mix_sounds(sounds))

## 3. Same sign, different meaning

numpy changed the meaning of `+` and `*`. Compare lists and arrays:

In [ ]:
a = [1, 2, 3, 4]
b = [10, 20, 30, 40]
print(a + b)

x = np.array(a)
y = np.array(b)
print(x + y, x * y)

On lists, `+` joins. `a * b` is an error: two lists cannot be multiplied. On arrays, both work element by element. `np.array(a)` makes an array from a list: a function that makes a new value of a type is called a **constructor**.

Who decides what `+` means? The **type** of the value. Every value has one:

In [ ]:
print(type(3), type("ciao"), type(a), type(x))

A type comes with functions attached. `type(3).__add__` is the function that adds integers:

In [ ]:
print(type(3).__add__(10, 20))
print((3).__add__(2))

Behind every operator there is a method call: `3 + 2` is short for `(3).__add__(2)`. Python asks the value **on the left** what `+` means for it. `__add__` is `+`, `__mul__` is `*`, `__sub__` is `-`, `__eq__` is `==`. `dir(3)` lists all the methods of an integer.

So if we could make a type of our own, we could decide what `+` means for it.

## 4. A class of our own

`class` creates a new type. The simplest useful class stores a value:

In [ ]:
class MyInt:
    def __init__(self, n):
        self.number = n

    def print(self):
        print(self.number)

x = MyInt(2)
print(type(x))
x.print()

- `MyInt(2)` makes a new **object** of type `MyInt`. Python calls `__init__` to fill it in.
- `self` is the object being made. `self.number = n` stores a value inside it: an **attribute**. There is no need to declare `number` anywhere else.
- A function written inside a class is a **method**. `x.print()` calls `print` with `x` as `self`.

Now `+`. `__add__` receives the object on the left as `self`, the one on the right as `other`:

In [ ]:
class MyInt:
    def __init__(self, n):
        self.number = n

    def __add__(self, other):
        return MyInt(self.number + other.number)

    def print(self):
        print(self.number)

y = MyInt(3) + MyInt(4)
print(type(y))
y.print()

Nothing forces `+` to add. If `__add__` returns `MyInt(self.number * other.number)`, then `MyInt(3) + MyInt(4)` prints 12. There is no reason to do that for numbers, but it shows that the author of a type chooses the meaning of its operators.

**Note:** Python has a shorter way to write classes that mostly store data, `@dataclass`. We will use it next time.

### A wave

The early lessons stored sounds as lists of samples. A class can give such a list the operators of numpy:

In [ ]:
class Wave:
    def __init__(self, samples):
        self.samples = list(samples)

    def peak(self):
        return max([abs(s) for s in self.samples])

    def __add__(self, other):
        n = max(len(self.samples), len(other.samples))
        a = self.samples + [0] * (n - len(self.samples))
        b = other.samples + [0] * (n - len(other.samples))
        return Wave([x + y for x, y in zip(a, b)])

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return Wave([other * x for x in self.samples])
        return NotImplemented

    def __repr__(self):
        return "Wave" + str(self.samples)

print(Wave([1, 2, 3]) + Wave([3, 4, 5, 6, 7]))
print((Wave([1, 2, 3]) + Wave([3, 4, 5, 6, 7])) * 0.25)

- `list(samples)` accepts anything that can become a list, an array or a `range` too.
- `peak` is a method that does not take extra arguments.
- `__add__` pads the shorter wave with zeros, then adds: mixing two sounds of different lengths. `zip` alone would stop at the shorter one and lose the end of the longer.
- `__mul__` multiplies by a number: a gain. `isinstance(other, (int, float))` asks whether `other` is an integer or a float. For anything else it returns `NotImplemented`, Python's way of saying "this case is not defined", and Python raises a `TypeError`.
- `__repr__` is what Python shows when it prints the object. Without it we would see something like `<__main__.Wave object at 0x10a3...>`.

In [ ]:
try:
    Wave([1, 2, 3]) * Wave([4, 5])
except TypeError as error:
    print("TypeError:", error)

`try` runs a line; if it raises a `TypeError`, Python runs the `except` part instead of stopping the notebook.

### Your turn · Multiplying two waves

Multiplying two waves sample by sample is what we did with `env * wave`: modulation. Extend `__mul__` so that `Wave * Wave` multiplies sample by sample, and `Wave * number` is still a gain. Use `zip`: the result is as long as the shorter wave.

In [ ]:
class Wave:
    def __init__(self, samples):
        self.samples = list(samples)

    def peak(self):
        return max([abs(s) for s in self.samples])

    def __add__(self, other):
        n = max(len(self.samples), len(other.samples))
        a = self.samples + [0] * (n - len(self.samples))
        b = other.samples + [0] * (n - len(other.samples))
        return Wave([x + y for x, y in zip(a, b)])

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return Wave([other * x for x in self.samples])
        if isinstance(other, Wave):
            return Wave([x * y for x, y in zip(self.samples, other.samples)])
        return NotImplemented

    def __repr__(self):
        return "Wave" + str(self.samples)

print(Wave([1, 2, 3]) * Wave([4, 5]))
print(Wave([1, 2, 3]) * 2)

**Note:** `w * 2` works, `2 * w` does not. Python asks the number `2` first, and a number does not know our `Wave`. Python then tries `w.__rmul__(2)` (r for *right*), which we have not written.

## 5. A pitch class

Since lesson 01 a pitch has been a bare number: a frequency in Hz, or semitones from a root. From now on we use the numbering of MIDI, the protocol of electronic instruments: one step is one semitone, middle C (C4) is 60, and **A4 is 69, 440 Hz**.

$$f = 440 \cdot 2^{(m - 69)/12}$$

What should we be able to do with pitches? Like dates and durations in a calendar, there are two kinds of values:

| operation | result | calendar |
|---|---|---|
| pitch + interval | pitch | date + 3 days → date |
| pitch − pitch | interval | date − date → days |
| pitch + pitch | meaningless | "today + tomorrow" |

An interval stays a plain number of semitones. A pitch becomes an object, and its class allows only the operations that make sense. We build it step by step.

In [ ]:
class Pitch:
    def __init__(self, midi):
        self.midi = midi

    def freq(self):
        return 440 * 2 ** ((self.midi - 69) / 12)

p = Pitch(69)
print(p.freq(), Pitch(70).freq())

`p + 1` fails: Python does not know what adding a number to a `Pitch` means. We tell it with `__add__`, and refuse pitch + pitch. `__repr__` shows the name of the note:

- `midi % 12` is the position in the octave, 0 for C, 9 for A;
- `midi // 12` (whole division) counts octaves. MIDI numbers octaves from −1, so the octave is `midi // 12 - 1`: 69 is A4.

In [ ]:
note_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

class Pitch:
    def __init__(self, midi):
        self.midi = midi

    def freq(self):
        return 440 * 2 ** ((self.midi - 69) / 12)

    def __add__(self, semitones):
        if isinstance(semitones, Pitch):
            raise TypeError("Add semitones, not a pitch.")
        return Pitch(self.midi + semitones)

    def __repr__(self):
        return note_names[self.midi % 12] + str(self.midi // 12 - 1)

A4 = Pitch(69)
B4 = A4 + 2
print(A4, B4, type(B4), B4.freq())

`A4` is just the name of a variable. And A4 plus B4?

In [ ]:
try:
    A4 + B4
except TypeError as error:
    print("TypeError:", error)

The error is ours, with our message: the class refuses an operation that has no meaning.

### Pitches from names

We would like to write `Pitch("A4")`. A name is a string: the last character is the octave, the rest is the note. Slices take it apart:

In [ ]:
name = "A#4"
print(name[:-1], name[-1])
print(int(name[-1]), note_names.index(name[:-1]))

`name[:-1]` is all but the last character, `name[-1]` the last one. `note_names.index("A#")` is the position of `"A#"` in the list, 10. The MIDI number is `12 * (octave + 1) + position`.

The constructor can accept either a number or a name: `isinstance(value, str)` asks whether the value is a string. We also add `-`: pitch − pitch is an interval, pitch − number a lower pitch.

In [ ]:
class Pitch:
    def __init__(self, value):
        if isinstance(value, str):           # a name like "C4": the octave last, the note before
            value = 12 * (int(value[-1]) + 1) + note_names.index(value[:-1])
        self.midi = value

    def freq(self):
        return 440 * 2 ** ((self.midi - 69) / 12)

    def __add__(self, semitones):
        if isinstance(semitones, Pitch):
            raise TypeError("Add semitones, not a pitch.")
        return Pitch(self.midi + semitones)

    def __sub__(self, other):
        if isinstance(other, Pitch):
            return self.midi - other.midi    # an interval
        return Pitch(self.midi - other)      # a lower pitch

    def __eq__(self, other):
        return isinstance(other, Pitch) and self.midi == other.midi

    def __repr__(self):
        return note_names[self.midi % 12] + str(self.midi // 12 - 1)

print(Pitch("A4"), Pitch("A4").midi, (Pitch("A4") + 1).freq())
print(Pitch("B5") - 10, Pitch("G4") - Pitch("C4"))
print(Pitch("A4") == Pitch(69))

Names handle octaves 0–9 and sharps only.

### Your turn · Equal or the same?

Comment out the `__eq__` method, run the class cell again, and try `Pitch("A4") == Pitch(69)`. What does Python compare when a class has no `__eq__`?

Without `__eq__`, `==` asks whether the two names point to **the same object**. Two different objects with the same `midi` are not the same object, so the answer is `False`. With `__eq__`, two pitches are equal when their notes are. Put it back before going on.

## 6. Scales from intervals

A scale is a tonic plus a list of intervals. Adding each interval to a `Pitch` gives the notes, with their names:

In [ ]:
major_intervals = [0, 2, 4, 5, 7, 9, 11, 12]
minor_melodic_intervals = [0, 2, 3, 5, 7, 9, 11, 12]

print([Pitch("G4") + i for i in major_intervals])
print([Pitch("G4") + i for i in minor_melodic_intervals])

G major has one sharp, F#: the names confirm it. From pitches to frequencies, from frequencies to sounds:

In [ ]:
g_major = [(i * 0.25, pluck((Pitch("G4") + major_intervals[i]).freq())) for i in range(len(major_intervals))]
play(mix_sounds(g_major))

Two lists of sounds can be joined with `+`. Each sound keeps its own onset, so two scales that start at the same time are mixed together:

In [ ]:
g_sharp_major = [(i * 0.25, pluck((Pitch("G#4") + major_intervals[i]).freq())) for i in range(len(major_intervals))]
play(mix_sounds(g_major + g_sharp_major))

### Your turn · Up and down

Play G melodic minor going up and then coming down. `intervals[::-1]` is the list reversed. Do not repeat the top note.

In [ ]:
up_down = minor_melodic_intervals + minor_melodic_intervals[::-1][1:]
g_minor = [(i * 0.25, pluck((Pitch("G4") + up_down[i]).freq())) for i in range(len(up_down))]
print([Pitch("G4") + i for i in up_down])
play(mix_sounds(g_minor))

## 7. What we would like to write

We have pitches. A **note** is a pitch with a duration. We would like to write melodies like this:

```python
Note(Pitch("A4"), 2) >> Note(Pitch("B4"))     # A4 for 2 beats, then B4
Note(Pitch("A4")) & Note(Pitch("C#5"))        # A4 and C#5 together: a chord
```

`>>` would mean *then*, `&` *together*. To get there we need a type for notes, a type for sequences of notes, and the meaning of the two operators. That is the next lesson, and with it a small ambient piece of long drones.

## What we built

- numpy applies arithmetic to every element: a tone, an envelope and a note in one line each, a mixer with one slice.
- An operator is a method call: `a + b` is `a.__add__(b)`. The type of `a` decides what `+` means.
- A class makes a new type. `__init__` stores its data in `self`, methods give it behaviour, special methods give it operators and a way to print itself.
- A good class allows the operations that make sense and refuses the others: pitch + interval, not pitch + pitch.

Before sharing your work, restart the notebook and run it in order.